# D162 — Common Table Expressions (CTEs) in MySQL

A Common Table Expression, usually called a **CTE**, gives a temporary name to the result of a query. The name can be used by the main statement that follows it.

```sql
WITH cte_name AS (
    SELECT ...
)
SELECT ...
FROM cte_name;
```

A CTE is useful when a query has several logical steps. It helps us name each step instead of placing queries inside several pairs of brackets.

## 1. Connect to MySQL

This lesson creates a small demonstration table and removes it at the end. A later section also creates a session-scoped temporary table so the two approaches can be compared.

In [ ]:
import os
import mysql.connector

connection = mysql.connector.connect(
    host=os.environ.get('MYSQL_HOSTNAME', '127.0.0.1'),
    port=int(os.environ.get('MYSQL_PORT', '3306')),
    user=os.environ.get('MYSQL_USERNAME', 'root'),
    password=os.environ.get('MYSQL_PASSWORD', 'root'),
    database=os.environ.get('MYSQL_DATABASE', 'olist_import_lab'),
)
print('Connected:', connection.is_connected())

In [ ]:
def execute_sql(sql, params=None):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if not cursor.with_rows:
            connection.commit()
            print(f'Statement completed. Affected rows: {cursor.rowcount}')
            return []
        columns = [column[0] for column in cursor.description]
        rows = cursor.fetchall()
        values = [['NULL' if value is None else str(value) for value in row] for row in rows]
        widths = [len(column) for column in columns]
        for row in values:
            widths = [max(width, len(value)) for width, value in zip(widths, row)]
        print(' | '.join(c.ljust(w) for c, w in zip(columns, widths)))
        print('-+-'.join('-' * w for w in widths))
        for row in values:
            print(' | '.join(v.ljust(w) for v, w in zip(row, widths)))
        return rows
    finally:
        cursor.close()

## 2. Create a simple 10-row sales table

Each row is one sale. An employee can have several rows. The repeated employee names make grouping useful.

In [ ]:
execute_sql('DROP TABLE IF EXISTS cte_demo_sales')
execute_sql("""
CREATE TABLE cte_demo_sales (
    sale_id INT PRIMARY KEY,
    employee_name VARCHAR(20) NOT NULL,
    region VARCHAR(10) NOT NULL,
    sale_date DATE NOT NULL,
    amount DECIMAL(10,2) NOT NULL
)
""")
execute_sql("""
INSERT INTO cte_demo_sales VALUES
(1,  'Asha',  'South', '2025-01-05', 1200.00),
(2,  'Asha',  'South', '2025-01-18',  800.00),
(3,  'Bilal', 'North', '2025-01-07', 1500.00),
(4,  'Bilal', 'North', '2025-02-04', 1800.00),
(5,  'Chen',  'South', '2025-01-22',  700.00),
(6,  'Chen',  'South', '2025-02-11',  900.00),
(7,  'Divya', 'North', '2025-02-15', 2100.00),
(8,  'Divya', 'North', '2025-03-02', 1100.00),
(9,  'Eshan', 'South', '2025-03-08',  600.00),
(10, 'Eshan', 'South', '2025-03-20', 1000.00)
""")
execute_sql('SELECT * FROM cte_demo_sales ORDER BY sale_id')

## 3. A query without a CTE

The first task is simple: total the sales for each employee. No CTE is needed yet.

In [ ]:
execute_sql("""
SELECT employee_name, COUNT(*) AS sale_count, SUM(amount) AS total_sales
FROM cte_demo_sales
GROUP BY employee_name
ORDER BY total_sales DESC
""")

## 4. When the query becomes harder to read

Now find employees whose total sales are above the average employee total. SQL must first calculate one total per employee and then calculate the average of those totals.

Without a CTE, a **derived table** can be used. A derived table is a subquery placed in the `FROM` clause. Notice that the same employee-total calculation appears twice.

In [ ]:
execute_sql("""
SELECT totals.employee_name, totals.total_sales
FROM (
    SELECT employee_name, SUM(amount) AS total_sales
    FROM cte_demo_sales
    GROUP BY employee_name
) AS totals
WHERE totals.total_sales > (
    SELECT AVG(all_totals.total_sales)
    FROM (
        SELECT employee_name, SUM(amount) AS total_sales
        FROM cte_demo_sales
        GROUP BY employee_name
    ) AS all_totals
)
ORDER BY totals.total_sales DESC
""")

## 5. The same task with CTEs

The query is split into named steps:

1. `employee_totals` creates one row per employee.
2. `average_total` calculates one average from those employee rows.
3. The final query joins the two named results and applies the filter.

The CTE version is longer than a very simple query, but it avoids repeating the same calculation and makes each step visible.

In [ ]:
execute_sql("""
WITH employee_totals AS (
    SELECT employee_name, SUM(amount) AS total_sales
    FROM cte_demo_sales
    GROUP BY employee_name
),
average_total AS (
    SELECT AVG(total_sales) AS average_employee_sales
    FROM employee_totals
)
SELECT e.employee_name, e.total_sales,
       ROUND(a.average_employee_sales, 2) AS average_employee_sales
FROM employee_totals e
CROSS JOIN average_total a
WHERE e.total_sales > a.average_employee_sales
ORDER BY e.total_sales DESC
""")

## 6. One CTE can feed another CTE

Several CTEs are separated by commas after one `WITH`. A later CTE can read an earlier CTE. This creates a clear pipeline of small transformations.

The example below first totals employees, then totals regions from those employee totals, and finally adds the employee's percentage of the region total.

In [ ]:
execute_sql("""
WITH employee_totals AS (
    SELECT employee_name, region, SUM(amount) AS employee_sales
    FROM cte_demo_sales
    GROUP BY employee_name, region
),
region_totals AS (
    SELECT region, SUM(employee_sales) AS region_sales
    FROM employee_totals
    GROUP BY region
)
SELECT e.employee_name, e.region, e.employee_sales, r.region_sales,
       ROUND(100 * e.employee_sales / r.region_sales, 2) AS region_percent
FROM employee_totals e
JOIN region_totals r ON r.region = e.region
ORDER BY e.region, e.employee_sales DESC
""")

## 7. CTE scope and lifetime

A CTE exists only inside one SQL statement. It is not stored in the database and cannot be used by the next notebook cell.

```sql
WITH employee_totals AS (...)
SELECT * FROM employee_totals;  -- works here

SELECT * FROM employee_totals;  -- fails in a new statement
```

This short lifetime is useful. Helper names do not remain in the database, so a long analysis does not create a large collection of temporary views.

## 8. What if CTEs are not available?

MySQL 8.0 and later support CTEs. Older MySQL versions can use a derived table:

```sql
SELECT *
FROM (
    SELECT employee_name, SUM(amount) AS total_sales
    FROM cte_demo_sales
    GROUP BY employee_name
) AS employee_totals;
```

A derived table is usually the closest replacement when the result is needed once inside one statement. Deeply nested derived tables can become difficult to read, which is one reason CTEs were added.

## 9. Temporary table as an alternative

Use a temporary table when an intermediate result must be reused by several separate SQL statements in the same session. A temporary table can also have indexes. It must be created and populated, and it uses temporary storage.

A **session** is one open database connection. The temporary table is visible only to that connection and normally disappears when the connection closes.

In [ ]:
execute_sql('DROP TEMPORARY TABLE IF EXISTS temp_employee_totals')
execute_sql("""
CREATE TEMPORARY TABLE temp_employee_totals AS
SELECT employee_name, region, COUNT(*) AS sale_count, SUM(amount) AS total_sales
FROM cte_demo_sales
GROUP BY employee_name, region
""")
execute_sql('CREATE INDEX idx_temp_total_sales ON temp_employee_totals(total_sales)')

In [ ]:
# First statement using the temporary result
execute_sql("""
SELECT * FROM temp_employee_totals
ORDER BY total_sales DESC
""")

# A separate statement can use the same temporary result
execute_sql("""
SELECT region, SUM(total_sales) AS region_sales
FROM temp_employee_totals
GROUP BY region
ORDER BY region_sales DESC
""")

## 10. View as an alternative

A **view** is a named query stored in the database. An ordinary view stores the query definition, not a separate copy of the result. It can be used by later sessions and shared with other users who have permission.

A view is suitable for stable, reusable business logic. It is not ideal for every small step in one analysis. Creating many helper views can clutter the database, create naming problems, and make dependencies harder to follow. CTEs keep those one-query steps close to the query that uses them.

The example uses the permanent Olist table because MySQL does not allow a permanent view to refer to a temporary table. The view is removed after the demonstration.

In [ ]:
execute_sql("""
CREATE OR REPLACE VIEW demo_seller_item_totals AS
SELECT seller_id, COUNT(*) AS item_count, SUM(price) AS item_value
FROM olist_order_items
GROUP BY seller_id
""")
execute_sql("""
SELECT seller_id, item_count, ROUND(item_value, 2) AS item_value
FROM demo_seller_item_totals
ORDER BY item_value DESC
LIMIT 5
""")
execute_sql('DROP VIEW demo_seller_item_totals')

## 11. CTE, derived table, temporary table, or view?

| Need | Good choice | Lifetime |
|---|---|---|
| Name a step inside one statement | CTE | one statement |
| Support older MySQL or use one nested result once | derived table | one statement |
| Reuse calculated rows across several statements in one connection | temporary table | one session |
| Share stable query logic across sessions and users | view | until dropped |

A CTE is mainly a clarity and structure tool. It is not automatically faster than a subquery. MySQL's optimizer decides whether to merge or materialize eligible query parts. Use `EXPLAIN` when performance matters.

## 12. Recursive CTE: a result refers to itself

A **recursive CTE** repeatedly uses its previous result. It is useful for hierarchies, paths, and generated sequences.

It has two parts joined with `UNION ALL`:

- the anchor query creates the first row;
- the recursive query creates later rows and must have a stopping condition.

In [ ]:
execute_sql("""
WITH RECURSIVE numbers AS (
    SELECT 1 AS number_value
    UNION ALL
    SELECT number_value + 1
    FROM numbers
    WHERE number_value < 5
)
SELECT number_value FROM numbers
ORDER BY number_value
""")

## 13. Good CTE habits

- Use names that describe the result, such as `employee_totals`.
- Let one CTE perform one clear step.
- List only the columns needed by later steps.
- Keep filtering close to the step where it belongs.
- Do not add a CTE to a query that is already simple and clear.
- Use a temporary table when the same calculated data is needed across statements.
- Use a view for stable shared logic, not for every temporary analysis step.
- Give recursive CTEs a clear stopping condition.

In [ ]:
execute_sql('DROP TEMPORARY TABLE IF EXISTS temp_employee_totals')
execute_sql('DROP TABLE IF EXISTS cte_demo_sales')
if connection.is_connected():
    connection.close()
print('MySQL connection closed; temporary tables are gone.')